Import the dataset

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("yashdogra/speech-commands")

print("Path to dataset files:", path)

C:\Users\leyan\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: C:\Users\leyan\.cache\kagglehub\datasets\yashdogra\speech-commands\versions\1


## Train SVM model
extract 500 random samples from each class in a temporary directory and train a SVM model using pyAudioAnalysis.

In [27]:
import numpy as np
from pathlib import Path
import random
import shutil
import tempfile
from pyAudioAnalysis import audioTrainTest as aT
import class_adapter as ca
import csv

In [ ]:
# np.Inf was removed in the NumPy 2.0 release. Use np.inf instead.
if not hasattr(np, "Inf"):
    np.Inf = np.inf
if not hasattr(np, "NaN"):
    np.NaN = np.nan

## Split training set and test set

In [22]:
base = Path(path)

class_splits = {}
train_samples = 500
test_samples = 100

for folder in sorted(base.iterdir()):
    if folder.is_dir() and folder.name != "_background_noise_":
        speaker_groups = {}

        for wav in sorted(folder.glob("*.wav")):
            speaker_id = wav.stem.split("_nohash_")[0]
            speaker_groups.setdefault(speaker_id, []).append(wav)

        speakers = sorted(speaker_groups)
        if len(speakers) < test_samples:
            raise ValueError(f"{folder.name}: servono almeno {test_samples} speaker, trovati {len(speakers)}")

        test_speakers = speakers[-test_samples:]
        train_speakers = speakers[:-test_samples]

        train_selected = []
        round_index = 0
        while len(train_selected) < train_samples:
            made_progress = False
            for speaker_id in train_speakers:
                wavs = speaker_groups[speaker_id]
                if round_index < len(wavs):
                    train_selected.append(wavs[round_index])
                    made_progress = True
                    if len(train_selected) == train_samples:
                        break
            if not made_progress:
                break
            round_index += 1

        if len(train_selected) != train_samples:
            raise ValueError(f"{folder.name}: disponibili solo {len(train_selected)} campioni per il train")

        test_selected = [speaker_groups[speaker_id][0] for speaker_id in test_speakers]

        class_splits[folder.name] = {
            "train": train_selected,
            "test": test_selected,
        }

### Create folders for train and test sets

In [24]:
tmp_train_root = Path(tempfile.mkdtemp())
tmp_test_root = Path(tempfile.mkdtemp())

train_dirs = []
test_dirs = []

for class_name, split in class_splits.items():
    train_dir = tmp_train_root / class_name
    test_dir = tmp_test_root / class_name
    train_dir.mkdir(parents=True, exist_ok=True)
    test_dir.mkdir(parents=True, exist_ok=True)

    for wav in split["train"]:
        shutil.copy2(wav, train_dir / wav.name)

    for wav in split["test"]:
        shutil.copy2(wav, test_dir / wav.name)

    train_dirs.append(str(train_dir))
    test_dirs.append(str(test_dir))

#print(train_dirs)
#print(test_dirs)

Train the SVM model using pyAudioAnalysis. The model will be saved as "svm_model".

In [3]:
aT.extract_features_and_train(
    train_dirs,
    1.0, 1.0,
    aT.shortTermWindow, aT.shortTermStep,
    "svm",
    "svm_model",
    True
)

Analyzing file 1 of 500: C:\Users\leyan\AppData\Local\Temp\tmp5ci_yjpz\backward\017c4098_nohash_3.wav
Analyzing file 2 of 500: C:\Users\leyan\AppData\Local\Temp\tmp5ci_yjpz\backward\042a8dde_nohash_2.wav
Analyzing file 3 of 500: C:\Users\leyan\AppData\Local\Temp\tmp5ci_yjpz\backward\0585b66d_nohash_1.wav
Analyzing file 4 of 500: C:\Users\leyan\AppData\Local\Temp\tmp5ci_yjpz\backward\0585b66d_nohash_2.wav
Analyzing file 5 of 500: C:\Users\leyan\AppData\Local\Temp\tmp5ci_yjpz\backward\067f61e2_nohash_2.wav
Analyzing file 6 of 500: C:\Users\leyan\AppData\Local\Temp\tmp5ci_yjpz\backward\067f61e2_nohash_4.wav
Analyzing file 7 of 500: C:\Users\leyan\AppData\Local\Temp\tmp5ci_yjpz\backward\06f6c194_nohash_0.wav
Analyzing file 8 of 500: C:\Users\leyan\AppData\Local\Temp\tmp5ci_yjpz\backward\08ab231c_nohash_0.wav
Analyzing file 9 of 500: C:\Users\leyan\AppData\Local\Temp\tmp5ci_yjpz\backward\0a2b400e_nohash_2.wav
Analyzing file 10 of 500: C:\Users\leyan\AppData\Local\Temp\tmp5ci_yjpz\backward\0

C:\Users\leyan\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Param = 0.00100 - classifier Evaluation Experiment 2 of 3


C:\Users\leyan\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Param = 0.00100 - classifier Evaluation Experiment 3 of 3


...

C:\Users\leyan\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Param = 20.00000 - classifier Evaluation Experiment 3 of 3


C:\Users\leyan\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


		backward			bed			bird			cat			dog			down			eight			five			follow			forward			four			go			happy			house			learn			left			marvin			nine			no			off			on			one			right			seven			sheila			six			stop			three			tree			two			up			visual			wow			yes			zero		OVERALL
	C	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	PRE	REC	f1	ACC	f1
	0.001	30.9	42.9	35.9	30.4	48.6	37.4	56.7	47.5	51.7	45.2	50.6	47.8	26.1	31.3	28.5	18.9	17.6	18.2	48.2	51.5	49.8	34.7	24.3	28.6	40.2	43.3	41.7	29.5	37.3	32.9	47.4	36.7	41.4	33.6	20.7	25.6	50.0	47.9	48.9	57.1	60.9	59.0	34.4	34.4	34.4	36.2	38.2	37.2	23.5	34.8	28.1	32.9	34.0	33.4	25.8	25.6	25.7	42.0	41.2	41.6	40.7	21.6	28.2	28.9	25.7	27.2	34.9	19.0

C:\Users\leyan\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(


Load the trained SVM model if it exists.

In [3]:
aT.load_model("svm_model", False)

(SVC(C=np.float64(0.5), gamma='auto', kernel='linear', probability=True),
 array([ 1.39845485e-01,  2.10429123e-02,  2.95218848e+00,  2.32874943e-01,
         2.14017122e-01,  9.98740848e-01,  1.35850325e-02,  2.28372809e-01,
        -3.06885461e+01,  1.88989746e+00, -1.02263706e-01,  1.38699901e-01,
        -2.66173467e-02,  8.97639799e-02, -5.89108013e-03,  1.39509546e-02,
        -7.24662174e-04, -1.48524172e-02, -4.37037513e-02,  1.78364385e-03,
        -3.58414441e-02,  1.49071071e-02,  5.18127441e-03,  3.19314972e-02,
         1.25157235e-02,  2.24767443e-02,  9.71919166e-03,  4.42293803e-02,
         3.58749257e-03,  6.89048076e-03,  1.30803387e-02,  3.15308530e-02,
         5.88361031e-03,  2.65927080e-02, -8.77996965e-04,  3.53363604e-04,
         5.96159244e-03, -1.68049862e-03, -3.87805814e-04, -5.75743023e-03,
         5.92718925e-04, -2.50169244e-03,  1.24652150e-01,  1.91280011e-02,
        -3.23890045e-03,  5.26729906e-05, -1.24523473e-03,  1.64350919e-03,
        -7.441

In [29]:
results = {
    class_name: {"correct": 0, "wrong": 0, "total": 0}
    for class_name in class_splits
}

for class_name, split in class_splits.items():
    for wav in split["test"]:
        predicted_num = aT.file_classification(str(wav), "svm_model", "svm")[0]
        predicted = ca.adapter(predicted_num)
        results[class_name]["total"] += 1

        if predicted == class_name:
            results[class_name]["correct"] += 1
        else:
            results[class_name]["wrong"] += 1

print(results)


{'backward': {'correct': 63, 'wrong': 37, 'total': 100}, 'bed': {'correct': 65, 'wrong': 35, 'total': 100}, 'bird': {'correct': 65, 'wrong': 35, 'total': 100}, 'cat': {'correct': 61, 'wrong': 39, 'total': 100}, 'dog': {'correct': 48, 'wrong': 52, 'total': 100}, 'down': {'correct': 31, 'wrong': 69, 'total': 100}, 'eight': {'correct': 69, 'wrong': 31, 'total': 100}, 'five': {'correct': 37, 'wrong': 63, 'total': 100}, 'follow': {'correct': 43, 'wrong': 57, 'total': 100}, 'forward': {'correct': 42, 'wrong': 58, 'total': 100}, 'four': {'correct': 54, 'wrong': 46, 'total': 100}, 'go': {'correct': 26, 'wrong': 74, 'total': 100}, 'happy': {'correct': 54, 'wrong': 46, 'total': 100}, 'house': {'correct': 72, 'wrong': 28, 'total': 100}, 'learn': {'correct': 34, 'wrong': 66, 'total': 100}, 'left': {'correct': 45, 'wrong': 55, 'total': 100}, 'marvin': {'correct': 55, 'wrong': 45, 'total': 100}, 'nine': {'correct': 44, 'wrong': 56, 'total': 100}, 'no': {'correct': 35, 'wrong': 65, 'total': 100}, 'of

In [30]:
model_name = "svm_model"
csv_name = f"{model_name}_results.csv"

with open(csv_name, "w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["class_name", "correct", "wrong", "total", "accuracy"])

    for class_name, stats in results.items():
        total = stats["correct"] + stats["wrong"]
        accuracy = stats["correct"] / total if total else 0
        writer.writerow([class_name, stats["correct"], stats["wrong"], total, accuracy])